In [4]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [5]:
df = pd.read_csv("E-commerce Website Logs.csv",na_values=["--"], dtype={"age": "Int64"})
df.head()


,accessed_date,duration_(secs),network_protocol,ip,bytes,accessed_Ffom,age,gender,country,membership,language,sales,returned,returned_amount,pay_method
0,2017-03-14 17:43:57.172,2533,TCP,1.10.195.126,20100,Chrome,28,Female,CA,Normal,English,261.9600,No,0.0,Credit Card
1,2017-03-14 17:43:57.172,4034,TCP,1.1.217.211,20500,Mozilla Firefox,21,Male,AR,Normal,English,731.9400,No,0.0,Debit Card
2,2017-03-14 17:43:26.135,1525,TCP,1.115.198.107,90100,Mozilla Firefox,20,Male,PL,Normal,English,14.6200,No,0.0,Cash
3,2017-03-14 17:43:26.135,4572,TCP,1.121.152.143,100300,Mozilla Firefox,66,Female,IN,Normal,Spanish,957.5775,No,0.0,Credit Card
4,2017-03-14 18:17:09.005,3652,TCP,1.123.135.213,270200,Mozilla Firefox,53,Female,KR,Normal,Spanish,22.3680,No,0.0,Cash


In [6]:
# Particionando os dados por data - Parte 1
df['data'] = pd.to_datetime(df["accessed_date"], errors="coerce").dt.date
for data in df['data'].unique():
    df_data = df[df['data'] == data]
    df_data.to_csv(f"dados_brutos/{data}.csv", index=False)

In [ ]:
def inicializar_estado(datas):
    if not os.path.exists("gerenciamento/log_processamento.csv"):
        df_estado = pd.DataFrame({
            "data": datas,
            "status": "pendente",
            "ultima_execucao": None,
            "erro": None
        })
        df_estado.to_csv("gerenciamento/log_processamento.csv", index=False)

def atualizar_estado(data, status, erro=None):
    df_estado = pd.read_csv("gerenciamento/log_processamento.csv")

    df_estado.loc[df_estado["data"] == str(data), "status"] = status
    df_estado.loc[df_estado["data"] == str(data), "ultima_execucao"] = datetime.now()
    df_estado.loc[df_estado["data"] == str(data), "erro"] = erro

    df_estado.to_csv("gerenciamento/log_processamento.csv", index=False)

def verificar_erros(df, var):
    erros_lista = []
    df = df[["accessed_date",var]]
    for idx, row in df.iterrows():
        motivo = []

        # Erro de data invalida
        if pd.isna(row["accessed_date"]):
            motivo.append("data_invalida")

        # Erro de valores nulos
        if row.isnull().any():
            motivo.append("valor_nulo")
            
        if motivo:
            linha_erro = row.copy()
            linha_erro["motivo_erro"] = "; ".join(motivo)
            erros_lista.append(linha_erro)

    df_erros = pd.DataFrame(erros_lista)

    return df_erros

# Vistas de lote - Parte 2
def acessar_dados(data, fazer, var):
    # Acessa os dados brutos separados por data
    df = pd.read_csv(f"dados_brutos/{data}.csv")
    
    # Salvando os arquivos de gerenciamento
    erros = verificar_erros(df, var)
    salvar_quarentena(erros, data)
    df = df.drop(erros.index)
    try:
        if df.empty:
            atualizar_estado(data, "erro",
                f"{data} sem dados válidos para {var} realizando '{fazer}'")
            return

        if var not in df.columns:
            atualizar_estado(data, "erro", f"Variável {var} não existe")
            return
        elif fazer == "media":
            resultado = df[var].mean()

        elif fazer == 'contagem':
            resultado = df[var].count()

        elif fazer == 'maximo':
            resultado = df[var].max()

        elif fazer == 'minimo':
            resultado = df[var].min()

        elif fazer == 'mediana':
            resultado = df[var].median()

        elif fazer == 'desvio_padrao':
            resultado = df[var].std()

        elif fazer == 'variancia':
            resultado = df[var].var()

        elif fazer == 'moda':
            resultado = df[var].mode()[0]        
        
        # Verifica se o arquivo existe, se não cria ele com a coluna fazer dinamicamente
        if not os.path.exists("vistas_lote/vistas_de_lote.csv"):
            vista_lote = pd.DataFrame(columns=["data", "var"])
        else:
            vista_lote = pd.read_csv('vistas_lote/vistas_de_lote.csv')

        # Cria a máscara para encontrar a linha específica
        mascara = ((vista_lote['data'] == str(data)) & (vista_lote['var'] == var))

        if mascara.any():
            vista_lote.loc[mascara, fazer] = resultado
        else:
            nova_linha = pd.DataFrame([{"data": str(data), "var": var, fazer: resultado}])
            vista_lote = pd.concat([vista_lote, nova_linha], ignore_index=True)

        # Salva o arquivo final
        vista_lote.to_csv("vistas_lote/vistas_de_lote.csv", index=False)

        # Salva o arquivo final e altera o log
        vista_lote.to_csv("vistas_lote/vistas_de_lote.csv", index=False)
        atualizar_estado(data, "processado")
        
    except Exception as e:
        print(f"Erro ao processar {data}: {e}")
        atualizar_estado(data, "erro")


def salvar_quarentena(df_erros, data):
    if df_erros.empty:
        return

    # Salva linhas problemáticas
    df_erros.to_csv(
        "gerenciamento/quarentena_linhas_brutas.csv",
        mode='a',
        index=False,
        header=not os.path.exists("gerenciamento/quarentena_linhas_brutas.csv")
    )

    # Salva o arquivo quarentena_linha_por_arquivo
    quarentena_linha_por_arquivo = pd.DataFrame({
        "data": [data],
        "quantidade_erros": [len(df_erros)]
    })

    # Salva o arquivo quarentena_linha_por_arquivo
    quarentena_linha_por_arquivo.to_csv(
        "gerenciamento/quarentena_linhas_por_arquivo.csv",
        mode='a',
        index=False,
        header=not os.path.exists("gerenciamento/quarentena_linhas_por_arquivo.csv")
    )


In [12]:
# Inicializa o estado no log
inicializar_estado(df['data'].unique())

In [14]:
# Gerando todos as vistas de lote das variáveis numéricas
for data in df['data'].unique():
        #for comando in ["media", "contagem", "maximo", "minimo", "mediana", "desvio_padrao", "variancia", "moda"]:
        for comando in ["media", "contagem"]:
                #for colum in ["duration_(secs)", "bytes", "age", "sales", "returned_amount"]:
                for colum in ["duration_(secs)","age"]:
                        acessar_dados(data, comando, colum)

C:\Users\Yan\AppData\Local\Temp\ipykernel_1852\4148996342.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2017-03-20 sem dados válidos para age realizando 'media'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_estado.loc[df_estado["data"] == str(data), "erro"] = erro
C:\Users\Yan\AppData\Local\Temp\ipykernel_1852\4148996342.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2017-03-20 sem dados válidos para age realizando 'contagem'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_estado.loc[df_estado["data"] == str(data), "erro"] = erro
